In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -q segment-anything albumentations networkx scikit-image opencv-python-headless tqdm
import torch
print("GPUs:", torch.cuda.device_count())

In [ ]:
os.makedirs("/kaggle/working/sam_ckpts", exist_ok=True)
ckpt_path = "/kaggle/working/sam_ckpts/sam_vit_b_01ec64.pth"
if not os.path.exists(ckpt_path):
    !wget -q -O {ckpt_path} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
print("checkpoint MB:", os.path.getsize(ckpt_path) / 1e6)

In [ ]:
import glob, math, random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from segment_anything import sam_model_registry
import networkx as nx
from skimage.morphology import skeletonize
from scipy.ndimage import maximum_filter
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = "/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset"
IMG_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 4
SAM_CKPT = "/kaggle/working/sam_ckpts/sam_vit_b_01ec64.pth"
SAM_TYPE = "vit_b"
SAM_INPUT_SIZE = 1024

PRETRAINED_DECODER = "/kaggle/input/models/aforaarushi/finetunewt/pytorch/default/1/fpn_decoder (2).pth"

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/demo_outputs", exist_ok=True)

In [ ]:
train_dir = os.path.join(DATA_ROOT, "train")
if not os.path.exists(train_dir):
    train_dir = DATA_ROOT

sat_files = sorted(glob.glob(os.path.join(train_dir, "*_sat.jpg")))
assert len(sat_files) > 0, f"No *_sat.jpg under {train_dir}"

pairs = []
for sp in sat_files:
    mp = sp.replace("_sat.jpg", "_mask.png")
    if os.path.exists(mp):
        pairs.append((sp, mp))

random.shuffle(pairs)
n_val = max(1, int(0.1 * len(pairs)))
val_pairs = pairs[:n_val]
train_pairs = pairs[n_val:]
print("train:", len(train_pairs), "val:", len(val_pairs))

In [ ]:
def adaptive_clahe_gamma(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l_eq = clahe.apply(l)
    lab_eq = cv2.merge([l_eq, a, b])
    out = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB)

    mean_lum = out.mean() / 255.0
    gamma = float(np.clip(0.4 + mean_lum, 0.6, 1.2))
    inv_gamma = 1.0 / gamma
    table = (np.arange(256) / 255.0) ** inv_gamma * 255.0
    table = table.astype("uint8")
    out = cv2.LUT(out, table)
    return out

In [ ]:
train_tf = A.Compose([
    A.RandomCrop(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.15),
    A.RandomShadow(p=0.3),
    A.CoarseDropout(max_holes=4, max_height=48, max_width=48, fill_value=0, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.CenterCrop(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


class RoadDataset(Dataset):
    def __init__(self, pairs, transform, use_adaptive_enhance=True):
        self.pairs = pairs
        self.transform = transform
        self.use_adaptive_enhance = use_adaptive_enhance

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        sp, mp = self.pairs[idx]
        img = cv2.imread(sp, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.use_adaptive_enhance:
            img = adaptive_clahe_gamma(img)

        mask = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 128).astype(np.float32)

        h, w = img.shape[:2]
        pad_h = max(0, IMG_SIZE - h)
        pad_w = max(0, IMG_SIZE - w)
        if pad_h > 0 or pad_w > 0:
            img = cv2.copyMakeBorder(img, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)

        augmented = self.transform(image=img, mask=mask)
        img_t = augmented["image"]
        mask_t = augmented["mask"].unsqueeze(0).float()
        return img_t, mask_t


train_ds = RoadDataset(train_pairs, train_tf)
val_ds = RoadDataset(val_pairs, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
print("train batches:", len(train_loader), "val batches:", len(val_loader))

In [ ]:
sam = sam_model_registry[SAM_TYPE](checkpoint=SAM_CKPT)
sam_encoder = sam.image_encoder
for p in sam_encoder.parameters():
    p.requires_grad = False
sam_encoder.eval()

def resize_to_sam(x):
    return F.interpolate(x, size=(SAM_INPUT_SIZE, SAM_INPUT_SIZE), mode="bilinear", align_corners=False)

print("SAM encoder frozen, params:", sum(p.numel() for p in sam_encoder.parameters()))

In [ ]:
TAP_BLOCKS = [6, 11]

class SAMFeaturePyramidExtractor(nn.Module):
    def __init__(self, encoder, tap_blocks=TAP_BLOCKS):
        super().__init__()
        self.encoder = encoder
        self.tap_blocks = set(tap_blocks)
        self.max_tap = max(tap_blocks)

    @torch.no_grad()
    def forward(self, x):
        x = self.encoder.patch_embed(x)
        if self.encoder.pos_embed is not None:
            x = x + self.encoder.pos_embed
        feats = []
        for i, blk in enumerate(self.encoder.blocks):
            x = blk(x)
            if i in self.tap_blocks:
                feats.append(x.permute(0, 3, 1, 2).contiguous())
            if i == self.max_tap:
                break
        final = self.encoder.neck(x.permute(0, 3, 1, 2).contiguous())
        feats.append(final)
        return feats

feature_extractor = SAMFeaturePyramidExtractor(sam_encoder).to(DEVICE)
feature_extractor.eval()

In [ ]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k=3, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, k, padding=p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class FPNDecoder(nn.Module):
    def __init__(self, in_channels_list, fpn_channels=128, num_classes=1):
        super().__init__()
        self.lateral_convs = nn.ModuleList([nn.Conv2d(c, fpn_channels, 1) for c in in_channels_list])
        self.smooth_convs = nn.ModuleList([ConvBNReLU(fpn_channels, fpn_channels) for _ in in_channels_list])
        self.aux_heads = nn.ModuleList([nn.Conv2d(fpn_channels, num_classes, 1) for _ in in_channels_list])
        self.final_fuse = ConvBNReLU(fpn_channels * len(in_channels_list), fpn_channels)
        self.final_head = nn.Conv2d(fpn_channels, num_classes, 1)

    def forward(self, feats, out_size):
        laterals = [lc(f) for lc, f in zip(self.lateral_convs, feats)]
        for i in range(len(laterals) - 2, -1, -1):
            up = F.interpolate(laterals[i + 1], size=laterals[i].shape[-2:], mode="bilinear", align_corners=False)
            laterals[i] = laterals[i] + up
        smoothed = [sc(l) for sc, l in zip(self.smooth_convs, laterals)]
        aux_outs, fused = [], []
        for head, s in zip(self.aux_heads, smoothed):
            aux_logit = head(s)
            aux_logit = F.interpolate(aux_logit, size=out_size, mode="bilinear", align_corners=False)
            aux_outs.append(aux_logit)
            fused.append(F.interpolate(s, size=out_size, mode="bilinear", align_corners=False))
        fused_cat = torch.cat(fused, dim=1)
        fused_feat = self.final_fuse(fused_cat)
        final_logit = self.final_head(fused_feat)
        return final_logit, aux_outs, fused_feat


fpn_in_channels = [768, 768, 256]
decoder = FPNDecoder(fpn_in_channels, fpn_channels=128, num_classes=1).to(DEVICE)
print("decoder params:", sum(p.numel() for p in decoder.parameters()))

In [ ]:
def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return 1 - ((2*inter+eps)/(union+eps)).mean()

def iou_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter
    return 1 - ((inter+eps)/(union+eps)).mean()

def boundary_loss(pred, target):
    pred = torch.sigmoid(pred)
    lap = torch.tensor([[0,1,0],[1,-4,1],[0,1,0]], dtype=torch.float32, device=pred.device).view(1,1,3,3)
    pe = F.conv2d(pred, lap, padding=1).abs()
    te = F.conv2d(target, lap, padding=1).abs()
    return F.l1_loss(pe, te)

def soft_erode(img):
    p1 = -F.max_pool2d(-img, (3,1), (1,1), (1,0))
    p2 = -F.max_pool2d(-img, (1,3), (1,1), (0,1))
    return torch.min(p1, p2)

def soft_dilate(img):
    return F.max_pool2d(img, (3,3), (1,1), (1,1))

def soft_open(img):
    return soft_dilate(soft_erode(img))

def soft_skeletonize(img, iters=5):
    img1 = soft_open(img)
    skel = F.relu(img - img1)
    for _ in range(iters):
        img = soft_erode(img)
        img1 = soft_open(img)
        delta = F.relu(img - img1)
        skel = skel + F.relu(delta - skel*delta)
    return skel

def soft_cldice_loss(pred, target, eps=1e-6, iters=5):
    pred_p = torch.sigmoid(pred)
    skel_pred = soft_skeletonize(pred_p, iters)
    skel_true = soft_skeletonize(target, iters)
    tprec = (torch.sum(skel_pred*target)+eps)/(torch.sum(skel_pred)+eps)
    tsens = (torch.sum(skel_true*pred_p)+eps)/(torch.sum(skel_true)+eps)
    return 1 - 2*(tprec*tsens)/(tprec+tsens+eps)

def combined_loss(pred, target, aux_outs=None, w_dice=1.0, w_iou=0.5, w_boundary=0.3, w_cldice=0.5, w_aux=0.4):
    main = (w_dice*dice_loss(pred,target) + w_iou*iou_loss(pred,target)
            + w_boundary*boundary_loss(pred,target) + w_cldice*soft_cldice_loss(pred,target))
    aux_loss = 0.0
    if aux_outs is not None:
        for a in aux_outs:
            aux_loss = aux_loss + dice_loss(a,target) + iou_loss(a,target)
        aux_loss = aux_loss / len(aux_outs)
    return main + w_aux*aux_loss

def compute_iou_dice(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).astype(np.float32)
    inter = (pred_bin*target).sum()
    union = pred_bin.sum() + target.sum() - inter
    iou = inter/(union+1e-6)
    dice = 2*inter/(pred_bin.sum()+target.sum()+1e-6)
    return iou, dice

In [ ]:
decoder.load_state_dict(torch.load(PRETRAINED_DECODER))
decoder.eval()
print("loaded 10-epoch decoder weights as starting point for overnight continuation")

In [ ]:
optimizer = torch.optim.AdamW(decoder.parameters(), lr=1e-4, weight_decay=1e-4)  # lower LR, continuing not restarting
SEG_EPOCHS = 25   # ~16 min/epoch observed earlier -> ~6.5 hrs, leaves headroom in your 12hr window
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SEG_EPOCHS)
scaler = torch.amp.GradScaler('cuda')

best_val_loss = float("inf")

for epoch in range(SEG_EPOCHS):
    decoder.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"seg epoch {epoch+1}/{SEG_EPOCHS}")

    for imgs, masks in pbar:
        imgs = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            imgs_sam = resize_to_sam(imgs)
            with torch.no_grad():
                feats = feature_extractor(imgs_sam)
            logits, aux_outs, _ = decoder(feats, out_size=imgs.shape[-2:])
            loss = combined_loss(logits, masks, aux_outs)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    scheduler.step()
    avg_loss = running_loss / len(train_loader)

    decoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(DEVICE); masks = masks.to(DEVICE)
            feats = feature_extractor(resize_to_sam(imgs))
            logits, aux_outs, _ = decoder(feats, out_size=imgs.shape[-2:])
            val_loss += combined_loss(logits, masks, aux_outs).item()
    val_loss /= len(val_loader)

    print(f"epoch {epoch+1}/{SEG_EPOCHS} train_loss={avg_loss:.4f} val_loss={val_loss:.4f}")

    torch.save(decoder.state_dict(), f"/kaggle/working/checkpoints/fpn_decoder_epoch{epoch+1}.pth")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(decoder.state_dict(), "/kaggle/working/fpn_decoder_best.pth")

    torch.cuda.empty_cache()
    import gc; gc.collect()

print("segmentation training complete, best val_loss:", best_val_loss)

In [ ]:
decoder.load_state_dict(torch.load("/kaggle/working/fpn_decoder_best.pth"))
decoder.eval()
all_probs, all_masks = [], []
with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc="final seg eval"):
        imgs = imgs.to(DEVICE)
        feats = feature_extractor(resize_to_sam(imgs))
        logits, _, _ = decoder(feats, out_size=imgs.shape[-2:])
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_masks.append(masks.numpy())

ious, dices = [], []
for probs, masks_np in zip(all_probs, all_masks):
    for b in range(probs.shape[0]):
        iou, dice = compute_iou_dice(probs[b,0], masks_np[b,0])
        ious.append(iou); dices.append(dice)
print(f"mean IoU: {np.mean(ious):.4f}, mean Dice: {np.mean(dices):.4f}")

In [ ]:
def extract_nodes_from_mask(prob_mask, threshold=0.5, min_distance=8):
    binary = (prob_mask > threshold).astype(np.uint8)
    skel = skeletonize(binary).astype(np.uint8)
    kernel = np.array([[1,1,1],[1,10,1],[1,1,1]])
    neighbor_count = cv2.filter2D(skel.astype(np.float32), -1, kernel.astype(np.float32))
    node_candidates = ((neighbor_count >= 13) | ((neighbor_count >= 11) & (neighbor_count < 12))) & (skel == 1)
    node_candidates = node_candidates.astype(np.uint8)
    dist_map = cv2.distanceTransform(node_candidates, cv2.DIST_L2, 3) + node_candidates.astype(np.float32)
    local_max = maximum_filter(dist_map, size=min_distance) == dist_map
    nodes_mask = local_max & (node_candidates > 0)
    ys, xs = np.where(nodes_mask)
    return list(zip(xs.tolist(), ys.tolist())), skel


class DisjointSet:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb
            return True
        return False


def heal_graph(nodes, existing_edges, max_gap_dist=40, angle_thresh_deg=35):
    n = len(nodes)
    ds = DisjointSet(n)
    for i, j in existing_edges:
        ds.union(i, j)
    adj = {i: [] for i in range(n)}
    for i, j in existing_edges:
        adj[i].append(j); adj[j].append(i)

    def local_direction(idx):
        if not adj[idx]:
            return None
        nx_, ny_ = nodes[idx]
        dirs = [np.array([nodes[nb][0]-nx_, nodes[nb][1]-ny_]) for nb in adj[idx]]
        v = np.mean(dirs, axis=0)
        norm = np.linalg.norm(v)
        return v/norm if norm > 1e-6 else None

    healed_edges = list(existing_edges)
    pts = np.array(nodes, dtype=np.float32)

    for i in range(n):
        ri = ds.find(i)
        candidates = [(np.linalg.norm(pts[i]-pts[j]), j) for j in range(n)
                      if ds.find(j) != ri and np.linalg.norm(pts[i]-pts[j]) <= max_gap_dist]
        candidates.sort(key=lambda t: t[0])
        dir_i = local_direction(i)
        for d, j in candidates[:5]:
            ok_angle = True
            if dir_i is not None:
                cand_vec = pts[j]-pts[i]
                cand_norm = np.linalg.norm(cand_vec)
                if cand_norm > 1e-6:
                    cosang = np.dot(dir_i, cand_vec/cand_norm)
                    angle = np.degrees(np.arccos(np.clip(cosang, -1, 1)))
                    ok_angle = angle <= angle_thresh_deg
            if ok_angle:
                healed_edges.append((i, j))
                ds.union(i, j)
                break
    return healed_edges

In [ ]:
def sample_features_at_points(feat_map, points, feat_size):
    img_h, img_w = feat_size
    pts = torch.tensor(points, dtype=torch.float32, device=feat_map.device)
    norm_x = (pts[:,0]/max(img_w-1,1))*2 - 1
    norm_y = (pts[:,1]/max(img_h-1,1))*2 - 1
    grid = torch.stack([norm_x, norm_y], dim=-1).view(1,-1,1,2)
    sampled = F.grid_sample(feat_map, grid, align_corners=True)
    return sampled.squeeze(0).squeeze(-1).permute(1,0)


def extended_line_points(p1, p2, extension_ratio=0.5, num_samples=8):
    p1 = np.array(p1, dtype=np.float32); p2 = np.array(p2, dtype=np.float32)
    direction = p2 - p1
    length = np.linalg.norm(direction)
    if length < 1e-6:
        return [tuple(p1)]*num_samples
    unit = direction/length
    ext = length*extension_ratio
    start = p1 - unit*ext
    end = p2 + unit*ext
    return [tuple(start + (end-start)*t) for t in np.linspace(0,1,num_samples)]


class ConnectivityClassifier(nn.Module):
    def __init__(self, feat_dim=128, hidden=256):
        super().__init__()
        in_dim = feat_dim*3
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(inplace=True), nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2), nn.ReLU(inplace=True),
            nn.Linear(hidden//2, 1),
        )
    def forward(self, node_a_feat, node_b_feat, line_feats):
        line_pooled = line_feats.mean(dim=1)
        x = torch.cat([node_a_feat, node_b_feat, line_pooled], dim=-1)
        return self.net(x).squeeze(-1)

In [ ]:
def build_connectivity_training_batch(prob_mask_np, gt_mask_np, num_pos=12, num_neg=12, snap_radius=10):
    pred_nodes, pred_skel = extract_nodes_from_mask(prob_mask_np)
    gt_nodes, gt_skel = extract_nodes_from_mask(gt_mask_np)
    if len(pred_nodes) < 2 or len(gt_nodes) < 2:
        return None

    pred_arr = np.array(pred_nodes)
    def snap_to_pred(gt_pt):
        d = np.linalg.norm(pred_arr - np.array(gt_pt), axis=1)
        idx = np.argmin(d)
        return tuple(pred_arr[idx]) if d[idx] <= snap_radius else gt_pt

    resampled_gt_nodes = [snap_to_pred(p) for p in gt_nodes]
    n = len(resampled_gt_nodes)

    skel_labels = cv2.connectedComponents(gt_skel.astype(np.uint8), connectivity=8)[1]
    def component_of(pt):
        x, y = int(round(pt[0])), int(round(pt[1]))
        x = np.clip(x, 0, skel_labels.shape[1]-1); y = np.clip(y, 0, skel_labels.shape[0]-1)
        return skel_labels[y, x]
    node_components = [component_of(p) for p in resampled_gt_nodes]

    pos_pairs, neg_pairs = [], []
    half_pos = num_pos // 2
    same_comp_pairs = [(i,j) for i in range(n) for j in range(i+1,n)
                        if node_components[i]==node_components[j] and node_components[i]!=0]
    random.shuffle(same_comp_pairs)
    for i, j in same_comp_pairs[:half_pos]:
        pos_pairs.append((resampled_gt_nodes[i], resampled_gt_nodes[j], 1))

    if len(same_comp_pairs) > half_pos:
        extra = same_comp_pairs[half_pos:half_pos+(num_pos-half_pos)*5]
        extra.sort(key=lambda ij: -np.linalg.norm(np.array(resampled_gt_nodes[ij[0]])-np.array(resampled_gt_nodes[ij[1]])))
        for i, j in extra[:num_pos-half_pos]:
            pos_pairs.append((resampled_gt_nodes[i], resampled_gt_nodes[j], 1))

    half_neg = num_neg // 2
    cross_comp_pairs = [(i,j) for i in range(n) for j in range(i+1,n) if node_components[i]!=node_components[j]]
    random.shuffle(cross_comp_pairs)
    for i, j in cross_comp_pairs[:half_neg]:
        neg_pairs.append((resampled_gt_nodes[i], resampled_gt_nodes[j], 0))
    for _ in range(num_neg - half_neg):
        i = random.randrange(n); j = random.randrange(len(pred_nodes))
        neg_pairs.append((resampled_gt_nodes[i], pred_nodes[j], 0))

    if not pos_pairs or not neg_pairs:
        return None
    return pos_pairs + neg_pairs

In [ ]:
import gc

connectivity_model = ConnectivityClassifier(feat_dim=128).to(DEVICE)  # fresh, not loading the old 3-epoch biased weights
conn_optimizer = torch.optim.AdamW(connectivity_model.parameters(), lr=1e-3, weight_decay=1e-4)
bce_loss = nn.BCEWithLogitsLoss()

CONN_EPOCHS = 15
decoder.eval()
feature_extractor.eval()

for epoch in range(CONN_EPOCHS):
    connectivity_model.train()
    running = 0.0
    steps = 0
    pbar = tqdm(train_loader, desc=f"conn epoch {epoch+1}/{CONN_EPOCHS}")

    for batch_idx, (imgs, masks) in enumerate(pbar):
        imgs = imgs.to(DEVICE)
        masks_np_batch = masks.numpy()

        with torch.no_grad():
            feats = feature_extractor(resize_to_sam(imgs))
            logits, aux_outs, fused_feat = decoder(feats, out_size=imgs.shape[-2:])
            probs = torch.sigmoid(logits)

        for b in range(imgs.shape[0]):
            prob_np = probs[b,0].detach().cpu().numpy()
            gt_np = masks_np_batch[b,0]
            pairs = build_connectivity_training_batch(prob_np, gt_np)
            if pairs is None:
                continue

            node_a_pts = [p[0] for p in pairs]
            node_b_pts = [p[1] for p in pairs]
            labels = torch.tensor([p[2] for p in pairs], dtype=torch.float32, device=DEVICE)

            feat_a = sample_features_at_points(fused_feat[b:b+1], node_a_pts, imgs.shape[-2:])
            feat_b = sample_features_at_points(fused_feat[b:b+1], node_b_pts, imgs.shape[-2:])
            line_feats_list = [sample_features_at_points(fused_feat[b:b+1], extended_line_points(pa,pb), imgs.shape[-2:])
                                for pa, pb in zip(node_a_pts, node_b_pts)]
            line_feats = torch.stack(line_feats_list, dim=0)

            conn_optimizer.zero_grad()
            preds = connectivity_model(feat_a, feat_b, line_feats)
            loss = bce_loss(preds, labels)
            loss.backward()
            conn_optimizer.step()

            running += loss.item(); steps += 1
            del feat_a, feat_b, line_feats, preds, loss

        del feats, logits, aux_outs, fused_feat, probs
        if batch_idx % 50 == 0:
            torch.cuda.empty_cache()
        pbar.set_postfix(loss=running/max(steps,1))

    print(f"connectivity epoch {epoch+1}/{CONN_EPOCHS} loss={running/max(steps,1):.4f}")
    torch.save(connectivity_model.state_dict(), f"/kaggle/working/checkpoints/connectivity_epoch{epoch+1}.pth")
    torch.cuda.empty_cache(); gc.collect()

torch.save(connectivity_model.state_dict(), "/kaggle/working/connectivity_classifier_final.pth")
print("connectivity training complete")

In [ ]:
def build_final_graph(nodes, edges, edge_weights=None):
    G = nx.Graph()
    for idx, (x, y) in enumerate(nodes):
        G.add_node(idx, pos=(x, y))
    for k, (i, j) in enumerate(edges):
        w = edge_weights[k] if edge_weights is not None else np.linalg.norm(np.array(nodes[i])-np.array(nodes[j]))
        G.add_edge(i, j, weight=w)
    return G

def criticality_analysis(G):
    return nx.betweenness_centrality(G, weight="weight", normalized=True)

def resilience_index(G, top_k_nodes):
    def network_stats(graph):
        if graph.number_of_nodes() < 2:
            return 0.0, 1, 0
        num_components = nx.number_connected_components(graph)
        largest_cc = max(nx.connected_components(graph), key=len)
        largest_cc_size = len(largest_cc)
        sub = graph.subgraph(largest_cc).copy()
        avg_path = nx.average_shortest_path_length(sub, weight="weight") if sub.number_of_nodes() > 1 else 0.0
        return avg_path, num_components, largest_cc_size

    baseline_path, _, baseline_cc_size = network_stats(G)
    results = {}
    G_perturbed = G.copy()
    for node in top_k_nodes:
        if node in G_perturbed:
            G_perturbed.remove_node(node)
        perturbed_path, num_components, cc_size = network_stats(G_perturbed)
        fragmentation = 1 - (cc_size/baseline_cc_size) if baseline_cc_size > 0 else 0.0
        path_ratio = perturbed_path/baseline_path if baseline_path > 0 else 0.0
        results[node] = {"path_length_ratio": round(path_ratio,3), "fragmentation": round(fragmentation,3),
                          "num_components_after": num_components}
    return baseline_path, results

In [ ]:
def run_full_pipeline(image_path, threshold=0.5, conn_threshold=0.5, max_candidate_dist=300):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = adaptive_clahe_gamma(img)
    tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
                     A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
    x = tf(image=img)["image"].unsqueeze(0).to(DEVICE)

    decoder.eval(); feature_extractor.eval(); connectivity_model.eval()
    with torch.no_grad():
        feats = feature_extractor(resize_to_sam(x))
        logits, _, fused_feat = decoder(feats, out_size=x.shape[-2:])
        prob = torch.sigmoid(logits)[0,0].cpu().numpy()
        nodes, skel = extract_nodes_from_mask(prob, threshold=threshold)
        pts = np.array(nodes)
        candidate_pairs = [(i,j) for i in range(len(nodes)) for j in range(i+1,len(nodes))
                            if np.linalg.norm(pts[i]-pts[j]) < max_candidate_dist]
        edges = []
        if candidate_pairs:
            node_a_pts = [nodes[i] for i,j in candidate_pairs]
            node_b_pts = [nodes[j] for i,j in candidate_pairs]
            feat_a = sample_features_at_points(fused_feat, node_a_pts, x.shape[-2:])
            feat_b = sample_features_at_points(fused_feat, node_b_pts, x.shape[-2:])
            line_feats = torch.stack([sample_features_at_points(fused_feat, extended_line_points(pa,pb), x.shape[-2:])
                                       for pa, pb in zip(node_a_pts, node_b_pts)], dim=0)
            preds = torch.sigmoid(connectivity_model(feat_a, feat_b, line_feats))
            for (i,j), p in zip(candidate_pairs, preds.cpu().numpy()):
                if p > conn_threshold:
                    edges.append((i,j))

    healed_edges = heal_graph(nodes, edges, max_gap_dist=40)
    G = build_final_graph(nodes, healed_edges)
    centrality = criticality_analysis(G)
    top_nodes = sorted(centrality, key=centrality.get, reverse=True)[:5]
    baseline, resilience = resilience_index(G, top_nodes)
    return {"prob_mask": prob, "nodes": nodes, "graph": G, "centrality": centrality,
            "baseline_path_length": baseline, "resilience_results": resilience}


result = run_full_pipeline(val_pairs[1][0])
print(f"nodes={result['graph'].number_of_nodes()} edges={result['graph'].number_of_edges()}")
for node, stats in result['resilience_results'].items():
    print(f"  node {node}: {stats}")

In [ ]:
import matplotlib.pyplot as plt

for idx_try in [0, 1, 2]:
    result = run_full_pipeline(val_pairs[idx_try][0])
    fig, ax = plt.subplots(figsize=(8,8))
    ax.imshow(result["prob_mask"], cmap="gray")
    pos = {i:(x,y) for i,(x,y) in enumerate(result["nodes"])}
    nx.draw(result["graph"], pos=pos, ax=ax, node_size=12, node_color="red", edge_color="cyan", width=1.0)
    ax.set_title(f"Sample {idx_try}: nodes={result['graph'].number_of_nodes()}, edges={result['graph'].number_of_edges()}")
    plt.savefig(f"/kaggle/working/demo_outputs/sample_{idx_try}_graph.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
def connectivity_ratio_for_sample(image_path, raw_max_dist=25, heal_max_dist=120):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = adaptive_clahe_gamma(img)
    tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
                     A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
    x = tf(image=img)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feats = feature_extractor(resize_to_sam(x))
        logits, _, _ = decoder(feats, out_size=x.shape[-2:])
        prob = torch.sigmoid(logits)[0,0].cpu().numpy()
    nodes, _ = extract_nodes_from_mask(prob)
    pts = np.array(nodes)
    raw_edges = [(i,j) for i in range(len(nodes)) for j in range(i+1,len(nodes)) if np.linalg.norm(pts[i]-pts[j]) < raw_max_dist]
    G_raw = build_final_graph(nodes, raw_edges)
    healed_edges = heal_graph(nodes, raw_edges, max_gap_dist=heal_max_dist)
    G_healed = build_final_graph(nodes, healed_edges)
    cc_raw = max(len(c) for c in nx.connected_components(G_raw)) if G_raw.number_of_nodes() else 0
    cc_healed = max(len(c) for c in nx.connected_components(G_healed)) if G_healed.number_of_nodes() else 0
    return cc_raw, cc_healed, len(nodes)

ratios = []
for sp, mp in val_pairs[:10]:
    cc_raw, cc_healed, n = connectivity_ratio_for_sample(sp)
    if n > 0:
        ratios.append(cc_healed/max(cc_raw,1))
print(f"mean connectivity ratio improvement: {np.mean(ratios):.3f}x")